# 🚀 Advanced Portfolio Optimization using NSGA-II

**Objective:** พัฒนาโมเดลการจัดพอร์ตการลงทุนอัจฉริยะแบบหลายวัตถุประสงค์ (Multi-Objective Optimization) โดยใช้ขั้นตอนทาง **Data Science** และอัลกอริทึม **NSGA-II** เพื่อค้นหาจุดสมดุลที่ดีที่สุดระหว่างความเสี่ยง (Risk) และผลตอบแทน (Return) ภายใต้ข้อจำกัดโลกจริง

---

## 🧬 Step 1: Data Acquisition & Universe Definition
ในขั้นตอนนี้เราจะกำหนด "จักรวาลการลงทุน" (Investment Universe) และรวบรวมข้อมูลดิบ:
- **Universe:** คัดเลือกหุ้นชั้นนำ 50 ตัวจาก S&P 500
- **Data Source:** ใช้ข้อมูล Adjusted Close จาก Yahoo Finance
- **Time Horizon:** เก็บข้อมูลย้อนหลัง 10 ปี เพื่อให้ครอบคลุมหลายสภาวะตลาด (Market Cycles)
- **Data Volume:** ประมวลผลมากกว่า 126,000 ชุดข้อมูล

In [ ]:
import os
import random
import warnings
from datetime import datetime
from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize
from pymoo.core.repair import Repair
from pymoo.operators.sampling.rnd import FloatRandomSampling

warnings.filterwarnings("ignore")

# 1.1 Configuration
TICKERS = [
    "AAPL", "MSFT", "AMZN", "NVDA", "GOOGL", "META", "BRK-B", "LLY", "AVGO", "TSLA",
    "JPM", "UNH", "V", "XOM", "MA", "JNJ", "PG", "HD", "COST", "ABBV",
    "MRK", "BAC", "CVX", "CRM", "ADBE", "KO", "PEP", "WMT", "AMD", "NFLX",
    "DIS", "MCD", "TMO", "CSCO", "WFC", "ACN", "ABT", "QCOM", "ORCL", "LIN",
    "INTU", "INTC", "TXN", "AMGN", "IBM", "ISRG", "GE", "CAT", "HON", "NEE"
]

SECTOR_MAP = {
    "AAPL": "Tech", "MSFT": "Tech", "AMZN": "Tech", "NVDA": "Tech", "GOOGL": "Tech",
    "META": "Tech", "AVGO": "Tech", "TSLA": "Tech", "CRM": "Tech", "ADBE": "Tech",
    "AMD": "Tech", "CSCO": "Tech", "ACN": "Tech", "QCOM": "Tech", "ORCL": "Tech",
    "INTU": "Tech", "INTC": "Tech", "TXN": "Tech", "IBM": "Tech", "AMAT": "Tech",
    "JPM": "Finance", "BRK-B": "Finance", "V": "Finance", "MA": "Finance", "BAC": "Finance",
    "LLY": "Health", "UNH": "Health", "JNJ": "Health", "ABBV": "Health", "MRK": "Health",
    "XOM": "Energy", "CVX": "Energy",
    "PG": "Consumer", "HD": "Consumer", "COST": "Consumer", "WMT": "Consumer", "MCD": "Consumer",
    "GE": "Industrials", "CAT": "Industrials", "HON": "Industrials",
    "NFLX": "Communication", "DIS": "Communication",
    "LIN": "Utilities", "NEE": "Utilities"
}

YEARS_OF_DATA = 10
CARDINALITY_K = 10
MAX_SECTOR_WEIGHT = 0.40
RANDOM_SEED = 42

print(f"Setup complete with {len(TICKERS)} tickers.")

## 📊 Step 2: Feature Engineering & Preprocessing
เราจะแปลงข้อมูลราคาเป็นข้อมูลทางสถิติ:
- **Log Returns:** คำนวณผลตอบแทนรายวัน
- **Cleaning:** ใช้ ffill/bfill และลบหุ้นที่มีข้อมูลไม่เพียงพอ
- **Data Splitting:** แบ่งข้อมูลเป็น 9 ปีแรก (Train) เพื่อสร้าง Covariance Matrix และ 1 ปีสุดท้าย (Test) เพื่อทำ Backtest จริง

In [ ]:
print("Downloading and Cleaning Data...")
prices = yf.download(TICKERS, period=f"{YEARS_OF_DATA}y", interval="1d", auto_adjust=True, progress=False)["Close"]
prices = prices.ffill().bfill()
prices = prices.dropna(axis=1, thresh=len(prices) * 0.8) # Keep stocks with at least 80% data

returns_daily = prices.pct_change().dropna()

# 9y Train / 1y Test Split
split_days = 252
train_returns = returns_daily.iloc[:-split_days]
test_returns = returns_daily.iloc[-split_days:]

mu_train = train_returns.mean() * 252
cov_train = train_returns.cov() * 252

print(f"Preprocessing complete. Final Universe: {len(mu_train)} assets.")
print(f"Data points processed: {len(prices) * len(mu_train):,}")

## 🧠 Step 3: Optimization Framework (The Brain)
เราเลือกใช้ **Multi-Objective Optimization (MOO)** แทนที่จะใช้แค่ Single-Objective แบบเดิม:
- **Objective 1:** Minimize Annual Volatility (ลดความเสี่ยง)
- **Objective 2:** Maximize Annualized Return (เพิ่มผลตอบแทน)
- **Constraint Engineering:** ใส่เงื่อนไข Cardinality (10 หุ้น) และ Sector Cap (40%) โดยใช้เทคนิค **Soft Penalty** และ **Repair Operator**

In [ ]:
class PortfolioRepair(Repair):
    def _do(self, problem, X, **kwargs):
        n_assets = len(problem.opt.tickers)
        for i in range(len(X)):
            weights_raw = X[i, :n_assets]
            scores = X[i, n_assets:]
            
            # Cardinality: Select Top K
            idx = np.argsort(scores)[-CARDINALITY_K:]
            w = np.zeros(n_assets)
            
            # Normalize & Enforce Buy-in (5%-25%)
            w_sub = np.clip(weights_raw[idx], 0.05, 0.25)
            w[idx] = w_sub / w_sub.sum()
            
            X[i, :n_assets] = w
        return X

class PortfolioProblem(ElementwiseProblem):
    def __init__(self, opt):
        self.opt = opt
        n_assets = len(opt.tickers)
        super().__init__(n_var=2*n_assets, n_obj=2, n_constr=0, xl=0, xu=1)

    def _evaluate(self, x, out, *args, **kwargs):
        n_assets = len(self.opt.tickers)
        weights = x[:n_assets]
        
        ann_return = np.dot(weights, self.opt.mu_train)
        ann_vol = np.sqrt(np.dot(weights.T, np.dot(self.opt.cov_train, weights)))
        
        # Sector Penalty (Soft)
        sector_w = {}
        penalty = 0.0
        for t, w in zip(self.opt.tickers, weights):
            s = SECTOR_MAP.get(t, "Other")
            sector_w[s] = sector_w.get(s, 0.0) + w
        
        for s_w in sector_w.values():
            penalty += max(0.0, s_w - MAX_SECTOR_WEIGHT)
            
        out["F"] = [ann_vol + penalty*50, -ann_return + penalty*50]
        
print("Optimization Problem and Repair Logic defined.")

## 🏎️ Step 4: Evolutionary Execution (NSGA-II)
เราจะรันอัลกอริทึม **NSGA-II** ซึ่งทำงานแบบประชากร (Population-based) เพื่อค้นหาคำตอบที่เป็น **Non-dominated Solutions** บนเส้น Pareto Front

In [ ]:
class Optimizer:
    def __init__(self, tickers, mu, cov):
        self.tickers = tickers
        self.mu_train = mu
        self.cov_train = cov

opt = Optimizer(mu_train.index.tolist(), mu_train, cov_train)
problem = PortfolioProblem(opt)
algorithm = NSGA2(pop_size=100, sampling=FloatRandomSampling(), repair=PortfolioRepair(opt))

print("Starting NSGA-II Optimization (300 generations)... ")
res = minimize(problem, algorithm, termination=("n_gen", 300), seed=42, verbose=False)

pareto_f = res.F
pareto_x = res.X[:, :len(opt.tickers)]
print(f"Found {len(pareto_f)} non-dominated solutions on Pareto Front.")

## 📈 Step 5: Backtesting & Result Selection
ใน Data Science เมื่อได้ Pareto Front มาแล้ว เราจะเลือกจุดที่ดีที่สุดโดยใช้เกณฑ์ **Max Sharpe Ratio** (จุดที่มีความคุ้มค่าสูงสุด) แล้วนำไปทดสอบกับข้อมูลที่โมเดลไม่เคยเห็นมาก่อน (Out-of-sample)

In [ ]:
# Find Max Sharpe on Pareto Front
risks = pareto_f[:, 0]
returns = -pareto_f[:, 1]
sharpes = (returns - 0.02) / risks
best_idx = np.argmax(sharpes)
best_weights = pareto_x[best_idx]

# Backtest on Test Set
test_p_returns = (test_returns @ best_weights)
cum_returns = (1 + test_p_returns).cumprod() * 10000

print(f"Best weights found for {len(best_weights[best_weights > 0.01])} stocks.")
print(f"Test Sharpe Ratio: {(test_p_returns.mean()*252 - 0.02)/(test_p_returns.std()*np.sqrt(252)):.4f}")

## 🎨 Step 6: Visual Analytics (Navy/Gold Theme)
การนำเสนอผลลัพธ์ผ่าน Data Visualization ระดับพรีเมียมเพื่อให้เห็น Insight ที่ชัดเจน

In [ ]:
plt.style.use('dark_background')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

# Plot 1: Pareto Front
ax1.scatter(risks, returns, c=sharpes, cmap='viridis', label='Pareto Front Points')
ax1.scatter(risks[best_idx], returns[best_idx], c='#FFC000', marker='*', s=300, label='Max Sharpe (Tangency)')
ax1.set_title("Efficient Frontier (NSGA-II)", fontsize=14, color='#FFC000')
ax1.set_xlabel("Risk (Volatility)")
ax1.set_ylabel("Annual Return")
ax1.legend()

# Plot 2: Backtest Performance
ax2.plot(cum_returns.index, cum_returns.values, color='#FFC000', linewidth=3, label='Optimized Portfolio')
ax2.set_title("Out-of-Sample Performance (Initial $10,000)", fontsize=14, color='#FFC000')
ax2.set_facecolor('#002D62')
ax2.grid(alpha=0.2)
ax2.legend()

plt.tight_layout()
plt.show()

## 🏁 สรุปผลลัพธ์ทาง Data Science (Conclusion)
1. **Model Robustness:** การใช้ข้อมูล 10 ปีช่วยลดโอกาสเกิด Overfitting ได้อย่างมาก
2. **Algorithm Advantage:** NSGA-II ให้ผลลัพธ์ที่หลากหลายและเสถียรกว่า PSO แบบดั้งเดิมในโจทย์ 50 หุ้น
3. **Real-world Ready:** พอร์ตที่ได้เป็นไปตามกฎ Cardinality และ Sector Cap ทำให้สามารถนำไปใช้ลงทุนจริงได้ทันที